# Module 13: Placebo, Permutation and Falsification Tests

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Intermediate [Module 12](../../Intermediate/Notebooks/Module_12_Placebo_Tests.ipynb)
ran three placebos and they agreed. This module asks the question that one
could not: **what is the right null distribution**, and it turns out that a
common way of building one is wrong for this design.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

## 2. Two placebo nulls for the same estimate

The in space placebo, standard in the synthetic control literature, pretends
each untreated unit in turn was the treated one. The permutation null
reassigns the **whole treatment pattern** at random.

They are not the same thing when more than one unit was treated.

In [ ]:
real = fit(d, KEEP)[0]

one = []
for a in COMPARISON:
    rest = [x for x in COMPARISON if x != a]
    e = fit(d[d["agency_id"].isin([a] + rest)], [a])[0]
    one.append(e)
    print(f"  {NAME[a]:34s} pretended treated alone: {e:+6.1f}%")
one = np.array(one)

In [ ]:
rng = np.random.default_rng(21)
ids = sorted(d["agency_id"].unique())
four = np.array([fit(d, list(rng.choice(ids, len(KEEP), replace=False)))[0]
                 for _ in range(400)])

p1 = (np.sum(one <= real) + 1) / (len(one) + 1)
p4 = np.mean(four <= real)
print(f"  the real estimate: {real:+.1f}%\n")
print(f"  one agency null, 7 draws:    spread {one.std():.1f}, p = {p1:.2f}")
print(f"  four agency null, 400 draws: spread {four.std():.1f}, p = {p4:.3f}")

**The same estimate, p = 0.25 against one null and p = 0.030 against the
other.**

The one agency null is far more dispersed, because a single agency's estimate
is much noisier than an average of four. Comparing a four agency estimate
against a one agency null compares it to a reference distribution that is too
wide, and understates the evidence.

**The null must be built by pretending to treat the same number of units that
were actually treated.** In space placebos are correct for synthetic control
with one treated unit, and are the wrong null for a multi unit difference in
differences.

## 3. Falsification on a variable the program cannot touch

A falsification test differs from a placebo: the design is unchanged and the
**outcome** is one the treatment has no route to.

In [ ]:
s = d.copy()
s["settled"] = ((s["agency_id"].isin(KEEP)) & (s["period"] == "after")).astype(float)
s["phase"] = ((s["agency_id"].isin(KEEP)) & (s["period"] == "phase")).astype(float)
for lab, out, off in [("use of force per arrest", "n_uof", s["lo"]),
                      ("arrests", "n_arrests", None),
                      ("calls for service", "total_cfs", None)]:
    z = smf.glm(f"{out} ~ C(agency_id)+C(year_month)+settled+phase", s,
                family=sm.families.Poisson(), offset=off).fit()
    lo, hi = z.conf_int().loc["settled"]
    print(f"  {lab:24s} {pct(z.params['settled']):+7.2f}%  "
          f"[{pct(lo):+6.2f}, {pct(hi):+6.2f}]")

Arrests are untouched. Calls for service move 0.42 percent with an interval
excluding zero, which is detectable and not substantively a pathway.

**A falsification test with enormous power fails on noise**, and the response
is to report the size rather than the verdict.

## 4. How many placebos is too many

Running placebos is cheap, which means running many and reporting the
agreeable ones is also cheap. The protection is to fix the list in advance and
report it whole.

In [ ]:
pre = d[d["period"] == "before"].copy()
cuts = ["2020-07", "2020-11", "2021-03", "2021-07", "2021-11",
        "2022-03", "2022-07", "2022-11"]
sig = 0
rows = []
for cut in cuts:
    t = pre.copy()
    t["fake"] = ((t["agency_id"].isin(KEEP)) & (t["year_month"] >= cut)).astype(float)
    z = smf.glm("n_uof ~ C(agency_id) + C(year_month) + fake", t,
                family=sm.families.Poisson(), offset=t["lo"]).fit()
    lo, hi = z.conf_int().loc["fake"]
    rej = not (lo < 0 < hi)
    sig += rej
    rows.append({"fake date": cut, "estimate": f"{pct(z.params['fake']):+.1f}%",
                 "rejects at 5 percent": "YES" if rej else ""})
print(f"  {sig} of {len(cuts)} placebo dates reject at the 5 percent level")
print(f"  at 5 percent each, {0.05 * len(cuts):.1f} false rejections are expected "
      f"across {len(cuts)} tests by chance alone\n")
pd.DataFrame(rows).set_index("fake date")

None rejects here, and the arithmetic in the second line is the point. **Run
eight placebos at the five percent level and you expect 0.4 false
rejections**; run twenty and you expect one, which someone will then have to
explain away.

| Practice | Why |
|---|---|
| Fix the placebo list before looking | otherwise the choice is the result |
| Report every placebo run | including the ones that failed |
| State the expected number of false rejections | it is the denominator for any that do |
| Do not correct for multiplicity and then claim a pass | a corrected pass is weaker evidence, not stronger |

## Exercise

Build the permutation null a third way: restrict the reassignments to sets of
four that resemble the real treated group in size.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    size = profile.set_index("agency_id")["sworn_officers"]
    target = float(np.log(size[KEEP]).mean())
    pool = sorted(d["agency_id"].unique())
    rows = []
    for label, tol in [("any four agencies", None),
                       ("four whose mean log size is within 0.5", 0.5),
                       ("four whose mean log size is within 0.25", 0.25)]:
        draws, tries = [], 0
        while len(draws) < 200 and tries < 20000:
            tries += 1
            pick = list(rng.choice(pool, 4, replace=False))
            if tol is not None and abs(np.log(size[pick]).mean() - target) > tol:
                continue
            draws.append(fit(d, pick)[0])
        draws = np.array(draws)
        rows.append({"assignments considered": label,
                     "draws": len(draws),
                     "null spread": round(draws.std(), 1),
                     "p for the real estimate": round(float(np.mean(draws <= real)), 3)})
    display(pd.DataFrame(rows).set_index("assignments considered"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

Narrowing the set of assignments to those resembling the real treated group
narrows the null and changes the p value.

**Every permutation test contains this choice**, and it is usually left
implicit. "Any four agencies" treats an assignment of the four smallest
agencies as equally possible, which it was not: the program went to large,
high rate departments.

The honest presentation states the assignment set as part of the test, in the
same way a difference in differences states its comparison group. A p value
from a permutation test is a statement of the form **"unusual among
assignments of this kind"**, and the kind has to be named.

Note also the second column. Restricting the set shrinks the number of draws
available, so a tightly restricted permutation test can run out of
assignments before it runs out of precision, which is its own limit at twelve
units.

</details>

---

**Next:** [Module 14: Sensitivity Analysis and Partial Identification](Module_14_Sensitivity_And_Partial_Identification.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*